In [1]:
pip install pymerkle

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Prágai Bálint\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pymerkle import InmemoryTree as MerkleTree

tree1 = MerkleTree(algorithm='sha256', security=False)
tree2 = MerkleTree(algorithm='sha256', security=False)

In [3]:
index = tree1.append_entry(b'foo')   # leaf index


value = tree1.get_leaf(index)        # leaf hash
print(f'leaf index: {index}')
print(f'leaf hash: {value.hex()}')

leaf index: 1
leaf hash: 1d2039fa7971f4bf01a1c20cb2a3fe7af46865ca9cd9b840c2063df8fec4ff75


In [4]:
tree2.append_entry(b'foo')
tree2.append_entry(b'bar')


index = 2
value_1 = tree2.get_leaf(index)        # leaf hash
print(f'leaf index: {index}')
print(f'leaf hash: {value_1.hex()}')

leaf index: 2
leaf hash: 485904129bdda5d1b5fbc6bc4a82959ecfb9042db44dc08fe87e360b0a3f2501


In [17]:
from pymerkle import verify_inclusion

proof = tree2.prove_inclusion(1,2)
base = tree2.get_leaf(1)
root = tree2.get_state(2)

verify_inclusion(base, root, proof)

In [18]:

base = tree1.get_leaf(1)
root = tree2.get_state(2)

verify_inclusion(base, root, proof)

In [7]:
import pandas as pd

df = pd.read_csv('./generated_table_data/users.csv')

df.iloc[0]

id                                       1
user_full_name                      User 1
city                                London
gender                                male
age                                     66
email                    user1@example.com
phone                          35840958516
bookingtime            2025-01-01 09:00:00
complaint                Burning sensation
satisfaction_rating                      4
allergy                             Pollen
Name: 0, dtype: object

In [8]:
# if "geopolygon" in lower_name:
    #     return rng.choice(["FI-Uusimaa", "FI-Pirkanmaa", "SE-Stockholm", "DE-Berlin"])
    # if "age_range" in lower_name:
    #     return rng.choice(["under 18", "18-24", "25-40", "41-65", "66-79", "80+"])

def age_bucketer(age):
    match age:
        case age if age < 18:
            return 'under 18'
        case age if age < 25:
            return '18-25'
        case age if age < 40:
            return '25-39'
        case age if age < 66:
            return '40-65'
        case age if age < 80:
            return '66-79'
        case _:
            return '80+'
        
def geopolygon_bucketer(geopolygon):
    match geopolygon:
        case geopolygon if geopolygon == "Helsinki":
            return 'FI-Uusimaa'
        case geopolygon if geopolygon == "Tampere":
                return 'FI-Pirkanmaa'
        case geopolygon if geopolygon == "Stockholm":
            return 'SE-Stockholm'
        case geopolygon if geopolygon == "Berlin":
            return 'DE-Berlin'
        case geopolygon if geopolygon == "London":
            return 'UK-London'
        case geopolygon if geopolygon == "Austin":
            return 'US-Austin'
        case geopolygon if geopolygon == "Tokyo":
            return 'JP-Tokyo'
        case geopolygon if geopolygon == "Singapore":
            return 'SG-Singapore'
        case _:
            return 'other'

In [69]:
from pymerkle import BaseMerkleTree

class FixmeTree(BaseMerkleTree):

    def __init__(self, algorithm='sha256'):
        """
        Storage setup and superclass initialization
        """
        self.hashes = []

        super().__init__(algorithm)

    @staticmethod
    def preprocess_entry(data):
        """
        Preprocesses data entry before encoding
        """
        if isinstance(data, pd.Series):
            data = data.copy()
            data['age_range'] = age_bucketer(data['age'])
            data['geopolygon'] = geopolygon_bucketer(data['city'])
            data = data.drop(labels=['user_full_name', 'email', 'phone', 'age', 'city'])
            
        return data
    
    def _encode_entry(self, data):
        """
        Prepares data entry for hashing
        {age: 25, city: "Helsinki"} -> b'{"age_range": 18-25, "geopolygon": "FI-Uusimaa"}'
        """
        proc_data = str(data)
        encoded_data = proc_data.encode('utf-8')
        return encoded_data
        
    def _store_leaf(self, data, digest):
        """
        Stores data hash in a new leaf and returns index
        """
        self.hashes += [digest]

        return len(self.hashes)


    def _get_leaf(self, index):
        """
        Returns the hash stored by the leaf specified
        """
        value = self.hashes[index - 1]

        return value


    def _get_leaves(self, offset, width):
        """
        Returns hashes corresponding to the specified leaf range
        """
        values = self.hashes[offset: offset + width]

        return values


    def _get_size(self):
        """
        Returns the current number of leaves
        """
        return len(self.hashes)

In [ ]:
tree_medium = FixmeTree(algorithm='sha256')
data = FixmeTree.preprocess_entry(df.iloc[0])
print(data)
for i in range(len(data)):
    tree_medium.append_entry(data.iloc[i])

print(f'tree size: {tree_medium.get_size()}')
print(f'tree state: {tree_medium.get_state().hex()}')

1
tree size: 8
tree state: 5908decf242bb29d443cb6ccfec1c6a65ae867af160915f274c0f43b7bc7bd2f


In [87]:
prove_tree = FixmeTree(algorithm='sha256')
prove_tree.append_entry(1)

proof = tree_medium.prove_inclusion(1,8)
base = prove_tree.get_leaf(1)
root = tree_medium.get_state(8)

verify_inclusion(base, root, proof)

In [ ]:
tree_large = FixmeTree(algorithm='sha256')

for i in range(1000):
    data = FixmeTree.preprocess_entry(df.iloc[i])
    for j in range(len(data)):
        print(f'entry {i} leaf {j}')
        tree_large.append_entry(data[j])

print(f'tree size: {tree_large.get_size()}')
print(f'tree state: {tree_large.get_state().hex()}')

entry 0 leaf 0
entry 0 leaf 1
entry 0 leaf 2
entry 0 leaf 3
entry 0 leaf 4
entry 0 leaf 5
entry 0 leaf 6
entry 0 leaf 7
entry 1 leaf 0
entry 1 leaf 1
entry 1 leaf 2
entry 1 leaf 3
entry 1 leaf 4
entry 1 leaf 5
entry 1 leaf 6
entry 1 leaf 7
entry 2 leaf 0
entry 2 leaf 1
entry 2 leaf 2
entry 2 leaf 3
entry 2 leaf 4
entry 2 leaf 5
entry 2 leaf 6
entry 2 leaf 7
entry 3 leaf 0
entry 3 leaf 1
entry 3 leaf 2
entry 3 leaf 3
entry 3 leaf 4
entry 3 leaf 5
entry 3 leaf 6
entry 3 leaf 7
entry 4 leaf 0
entry 4 leaf 1
entry 4 leaf 2
entry 4 leaf 3
entry 4 leaf 4
entry 4 leaf 5
entry 4 leaf 6
entry 4 leaf 7
entry 5 leaf 0
entry 5 leaf 1
entry 5 leaf 2
entry 5 leaf 3
entry 5 leaf 4
entry 5 leaf 5
entry 5 leaf 6
entry 5 leaf 7
entry 6 leaf 0
entry 6 leaf 1
entry 6 leaf 2
entry 6 leaf 3
entry 6 leaf 4
entry 6 leaf 5
entry 6 leaf 6
entry 6 leaf 7
entry 7 leaf 0
entry 7 leaf 1
entry 7 leaf 2
entry 7 leaf 3
entry 7 leaf 4
entry 7 leaf 5
entry 7 leaf 6
entry 7 leaf 7
entry 8 leaf 0
entry 8 leaf 1
entry 8 le

In [104]:
tree_full = FixmeTree(algorithm='sha256')

for i in range(len(df)):
    data = FixmeTree.preprocess_entry(df.iloc[i])
    for j in range(len(data)):
        print(f'entry {i} leaf {j}')
        tree_full.append_entry(data[j])

print(f'tree size: {tree_full.get_size()}')
print(f'tree state: {tree_full.get_state().hex()}')

entry 0 leaf 0
entry 0 leaf 1
entry 0 leaf 2
entry 0 leaf 3
entry 0 leaf 4
entry 0 leaf 5
entry 0 leaf 6
entry 0 leaf 7
entry 1 leaf 0
entry 1 leaf 1
entry 1 leaf 2
entry 1 leaf 3
entry 1 leaf 4
entry 1 leaf 5
entry 1 leaf 6
entry 1 leaf 7
entry 2 leaf 0
entry 2 leaf 1
entry 2 leaf 2
entry 2 leaf 3
entry 2 leaf 4
entry 2 leaf 5
entry 2 leaf 6
entry 2 leaf 7
entry 3 leaf 0
entry 3 leaf 1
entry 3 leaf 2
entry 3 leaf 3
entry 3 leaf 4
entry 3 leaf 5
entry 3 leaf 6
entry 3 leaf 7
entry 4 leaf 0
entry 4 leaf 1
entry 4 leaf 2
entry 4 leaf 3
entry 4 leaf 4
entry 4 leaf 5
entry 4 leaf 6
entry 4 leaf 7
entry 5 leaf 0
entry 5 leaf 1
entry 5 leaf 2
entry 5 leaf 3
entry 5 leaf 4
entry 5 leaf 5
entry 5 leaf 6
entry 5 leaf 7
entry 6 leaf 0
entry 6 leaf 1
entry 6 leaf 2
entry 6 leaf 3
entry 6 leaf 4
entry 6 leaf 5
entry 6 leaf 6
entry 6 leaf 7
entry 7 leaf 0
entry 7 leaf 1
entry 7 leaf 2
entry 7 leaf 3
entry 7 leaf 4
entry 7 leaf 5
entry 7 leaf 6
entry 7 leaf 7
entry 8 leaf 0
entry 8 leaf 1
entry 8 le